# Exp 6C — Does Ultralytics' official mAP depend on the validation batch size?

The first run of Part C reported 0.000 mAP for every setting. That was a dataset-format bug, not a result: Ultralytics resolved the symlinked image folder to its real path and then looked for the labels in the wrong place ("0 images, 5000 backgrounds"). This notebook copies the images into a proper `images/val2017` + `labels/val2017` layout (hard links, so it takes no extra space and no time), clears any stale cache, and runs `model.val()` at batch 1, 8 and 32.

For each model it also checks that the labels were found: 5,000 images with ~36,000 instances. It stops if they weren't.

**Kaggle:** GPU on, Internet on. About 25–30 minutes.

In [ ]:
!pip install -q "ultralytics==8.4.172" pycocotools

In [ ]:
import os, io, glob, json, time, zipfile, shutil, contextlib, urllib.request, logging
import pandas as pd
from pycocotools.coco import COCO
logging.getLogger("ultralytics").setLevel(logging.ERROR)

OUT = "/kaggle/working/exp6c"; os.makedirs(OUT, exist_ok=True)
MODELS = ["yolo11s", "yolo11m", "yolo26s", "yolo26m"]
BATCHES = [1, 8, 32]

# COCO val2017 (attached or downloaded)
anns = glob.glob("/kaggle/input/**/instances_val2017.json", recursive=True)
probe = glob.glob("/kaggle/input/**/000000000139.jpg", recursive=True)
if anns and probe:
    ANN, SRC = anns[0], os.path.dirname(probe[0])
else:
    D = "/kaggle/working/coco"; os.makedirs(D, exist_ok=True)
    ANN, SRC = f"{D}/instances_val2017.json", f"{D}/val2017"
    if not os.path.exists(ANN):
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", f"{D}/a.zip")
        with zipfile.ZipFile(f"{D}/a.zip") as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
    if not (os.path.isdir(SRC) and len(os.listdir(SRC)) >= 5000):
        urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", f"{D}/v.zip")
        with zipfile.ZipFile(f"{D}/v.zip") as z:
            z.extractall(D)
with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(ANN)

In [ ]:
# Ultralytics layout: <root>/images/val2017/*.jpg and <root>/labels/val2017/*.txt (one line per box: cls xc yc w h, normalised)
ROOT = "/kaggle/working/yolo_coco_v2"
IMG, LAB = f"{ROOT}/images/val2017", f"{ROOT}/labels/val2017"
os.makedirs(IMG, exist_ok=True); os.makedirs(LAB, exist_ok=True)
for c in glob.glob(f"{ROOT}/**/*.cache", recursive=True):
    os.remove(c)
cats = sorted(GT.getCatIds()); cidx = {c: k for k, c in enumerate(cats)}
n_inst = 0
for im in GT.loadImgs(sorted(GT.getImgIds())):
    dst = f"{IMG}/{im['file_name']}"
    if not os.path.exists(dst):
        try:
            os.link(f"{SRC}/{im['file_name']}", dst)          # hard link: no copy, no extra space
        except OSError:
            shutil.copy(f"{SRC}/{im['file_name']}", dst)     # different file system (e.g. attached input)
    W, H = im["width"], im["height"]; lines = []
    for a in GT.loadAnns(GT.getAnnIds(imgIds=[im["id"]], iscrowd=False)):
        x, y, w, h = a["bbox"]
        if w > 0 and h > 0:
            lines.append(f"{cidx[a['category_id']]} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
    n_inst += len(lines)
    with open(f"{LAB}/{os.path.splitext(im['file_name'])[0]}.txt", "w") as f:
        f.write("\n".join(lines))
names = {k: GT.loadCats(c)[0]["name"] for k, c in enumerate(cats)}
YAML = f"{ROOT}/coco_val.yaml"
with open(YAML, "w") as f:
    f.write(f"path: {ROOT}\ntrain: images/val2017\nval: images/val2017\nnames:\n" + "".join(f"  {k}: {v}\n" for k, v in names.items()))
print(f"{len(os.listdir(IMG))} images, {len(os.listdir(LAB))} label files, {n_inst} boxes")

In [ ]:
from ultralytics import YOLO
rows = pd.read_csv(f"{OUT}/C_val.csv").to_dict("records") if os.path.exists(f"{OUT}/C_val.csv") else []
done = {(r["model"], r["batch"]) for r in rows}
for m in MODELS:
    for b in BATCHES:
        if (m, b) in done:
            continue
        t0 = time.time()
        res = YOLO(m + ".pt").val(data=YAML, batch=b, imgsz=640, conf=0.001, iou=0.7, rect=True, plots=False,
                                  verbose=False, device=0, workers=2)
        n_lab = int(res.nt_per_class.sum()) if hasattr(res, "nt_per_class") and res.nt_per_class is not None else -1
        assert n_lab != 0, "labels were not found again; stop and send me the log"
        rows.append({"model": m, "batch": b, "mAP50_95": 100 * res.box.map, "mAP50": 100 * res.box.map50,
                     "instances": n_lab, "minutes": (time.time() - t0) / 60})
        print(f"{m} batch={b:2d}: mAP50-95 {100 * res.box.map:.3f}  mAP50 {100 * res.box.map50:.3f}  (instances {n_lab})")
        pd.DataFrame(rows).to_csv(f"{OUT}/C_val.csv", index=False)

In [ ]:
R = pd.read_csv(f"{OUT}/C_val.csv")
piv = R.pivot_table(index="model", columns="batch", values="mAP50_95")
piv["range (max-min)"] = piv.max(axis=1) - piv.min(axis=1)
display(piv.style.format("{:.3f}"))
print("Upload exp6c/C_val.csv")